# Indexing demo
Preprocessing → inverted index → Boolean retrieval. Run the cells from top to bottom.
The code itself lives in the `indexing/` package; this notebook just calls it.

## 1. Load the corpus

In [1]:
import json

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

print(len(corpus), "chunks")
first = next(iter(corpus))
print(first, "->", corpus[first]["text"][:200])

789 chunks
siem_reap_01 -> Siem Reap – Overview. The town of Siem Reap (សៀមរាប), in northern Cambodia , is the primary access point for the Angkor Archaeological Park , just north of the city. It is quite laid-back and a pleasa


## 2. Preprocessing: tokenize, remove stop words, stem

In [2]:
from indexing.preprocess import tokenize, preprocess

text = "Taking the buses from Phnom Penh to Siem Reap is running daily."
print("tokens        :", tokenize(text))
print("no stop words :", preprocess(text, remove_stopwords=True, use_stemming=False))
print("stop + stem   :", preprocess(text, remove_stopwords=True, use_stemming=True))

tokens        : ['taking', 'the', 'buses', 'from', 'phnom', 'penh', 'to', 'siem', 'reap', 'is', 'running', 'daily']
no stop words : ['taking', 'buses', 'phnom', 'penh', 'siem', 'reap', 'running', 'daily']
stop + stem   : ['take', 'buse', 'phnom', 'penh', 'siem', 'reap', 'run', 'daili']


## 3. Build the inverted index

In [3]:
from indexing.inverted_index import InvertedIndex

idx = InvertedIndex.from_corpus(corpus)
print("chunks:", idx.N, "| unique terms:", len(idx.postings), "| avgdl: %.1f" % idx.avgdl)

term = idx.analyze("temple")[0]
print(term, "-> df =", idx.df(term))
print(list(idx.get_postings(term).items())[:5])

chunks: 789 | unique terms: 6679 | avgdl: 77.2
templ -> df = 127
[('siem_reap_01', 1), ('siem_reap_10', 1), ('siem_reap_17', 2), ('siem_reap_18', 2), ('siem_reap_21', 4)]


In [4]:
# Save so other modules (ranking, app) can load it without rebuilding
idx.save("data/index.json")

## 4. Boolean retrieval
Operators: `AND`, `OR`, `NOT`, parentheses. Words with no operator mean AND.

In [5]:
from indexing.boolean_retrieval import boolean_search

query = "bus AND (phnom penh OR siem reap) NOT taxi"
results = boolean_search(idx, query)
print(len(results), "matches")

for cid in results[:5]:
    d = corpus[cid]
    print(f"{cid} | {d['title']} - {d['section']}")
    print("   ", d["text"][:150], "...")

53 matches
angkor_30 | Angkor Archaeological Park - See – Outlying temples
    Angkor Archaeological Park – See – Outlying temples. 24 Beng Mealea . 80 km east of Siem Reap, about 1½ hr by car. Along with Ta Phrom and others, thi ...
anlong_veng_03 | Anlong Veng - Get in – By bus
    Anlong Veng – Get in – By bus. Next to the roundabout are the offices of G.S.T and Angkor Paramount Transport which both run buses to and from Siem Re ...
bangkok_to_ho_chi_minh_city_overland_08 | Bangkok to Ho Chi Minh City overland - Get in – Bangkok to Siem Reap
    Bangkok to Ho Chi Minh City overland – Get in – Bangkok to Siem Reap. Time: From 9–14 hours, depending on the season Cambodian bus company Giant Ibis  ...
bangkok_to_ho_chi_minh_city_overland_11 | Bangkok to Ho Chi Minh City overland - Get in – Siem Reap to Phnom Penh
    Bangkok to Ho Chi Minh City overland – Get in – Siem Reap to Phnom Penh. You have two options; road or boat. Which is more comfortable depends on the  ...
bangkok_to_ho_chi

## 5. Effect of stop words / stemming (for the report)

In [6]:
for stop in (False, True):
    for stem in (False, True):
        i = InvertedIndex.from_corpus(corpus, remove_stopwords=stop, use_stemming=stem)
        n = len(boolean_search(i, "temples AND visit"))
        print(f"stopwords={stop!s:5} stemming={stem!s:5} terms={len(i.postings):5} matches={n}")

stopwords=False stemming=False terms= 9327 matches=15
stopwords=False stemming=True  terms= 6775 matches=32
stopwords=True  stemming=False terms= 9208 matches=15
stopwords=True  stemming=True  terms= 6679 matches=32
